Analysis document for the stl to scan comparisions. Aiming to find the mean distance error, the method error (error between the default folders and "standards" folders), and the t test p value.

In [1]:
import pandas as pd
import sys
import os
import numpy as np

In [2]:
mast_df = pd.read_csv(r"D:\Projects\3D-textures\Experimental\scan_comparison\scan_comp_master_results.csv")
print(mast_df.head().to_string())

                                    filename printer  standards  stl_texture  test_no  avg_dist  std_dist  aligned
0  processed_models\BAMBU STANDARDS\B1T1.ply   BAMBU       True            1        1  0.479479  0.406014     True
1  processed_models\BAMBU STANDARDS\B1T2.ply   BAMBU       True            1        2  0.473799  0.386038     True
2  processed_models\BAMBU STANDARDS\B1T3.ply   BAMBU       True            1        3  0.491450  0.420577     True
3  processed_models\BAMBU STANDARDS\B1T4.ply   BAMBU       True            1        4  0.486166  0.414465     True
4  processed_models\BAMBU STANDARDS\B1T5.ply   BAMBU       True            1        5  0.402532  0.341339     True


In [39]:
# 1. Filter for aligned results only
df = mast_df[mast_df['aligned'] == True].copy()

# 2. Group by printer, texture, AND the standards flag
# This creates separate rows for standards=True and standards=False
summary_df = df.groupby(['printer', 'stl_texture', 'standards'])['avg_dist'].agg(['mean', 'std', 'count']).reset_index()

# 3. Calculate Standard Error (SE = std / sqrt(n))
summary_df['std_error'] = summary_df['std'] / np.sqrt(summary_df['count'])

# 4. Rename columns for clarity
# We rename 'std' to 'std_dev' to avoid confusion with the 'standards' column
summary_df = summary_df.rename(columns={
    'mean': 'mean_dist',
    'std': 'std_dev',
    'count': 'sample_size'
})

# 5. Display the results
# We include 'standards' so you can clearly see which row is which
final_df = summary_df.filter(['printer', 'stl_texture', 'standards', 'mean_dist', 'std_error', 'sample_size'])
print(final_df.sort_values(["standards", "printer"]))

   printer  stl_texture  standards  mean_dist  std_error  sample_size
0    BAMBU            1      False   0.504618   0.020527            6
2    BAMBU            2      False   0.945172   0.014682            6
4    BAMBU            3      False   0.624416   0.028655            5
6    BAMBU            4      False   0.917502   0.024029            6
8    BAMBU            5      False   0.566819   0.107587            6
10   BAMBU            6      False   0.801908   0.019018            6
12   ENDER            2      False   1.168778   0.003179            2
14   ENDER            3      False   1.072981   0.043491            5
16   ENDER            4      False   1.124693        NaN            1
17   ENDER            5      False   1.046613   0.022122            3
19   RESIN            1      False   0.719653   0.031641            4
21   RESIN            2      False   0.767099   0.035781            5
23   RESIN            3      False   0.591875   0.026120            4
25   RESIN          

In [5]:

# 1. Filter out any rows where avg_dist is over 1.5
df = mast_df[mast_df['avg_dist'] <= 1.5].copy()

# 2. Group by printer, texture, AND the standards flag
summary_df = df.groupby(['printer', 'stl_texture', 'standards'])['avg_dist'].agg(['mean', 'std', 'count']).reset_index()

# 3. Calculate Standard Error (SE = std / sqrt(n))
summary_df['std_error'] = summary_df['std'] / np.sqrt(summary_df['count'])

# 4. Rename columns for clarity
summary_df = summary_df.rename(columns={
    'mean': 'mean_dist',
    'std': 'std_dev',
    'count': 'sample_size'
})

# 5. Display the results
final_df = summary_df.filter(['printer', 'stl_texture', 'standards', 'mean_dist', 'std_error', 'sample_size']).copy()

# Round the float columns to 2 decimal places
final_df['mean_dist'] = final_df['mean_dist'].round(3)
final_df['std_error'] = final_df['std_error'].round(3)

# Sort by standards (True/False) then printer for a cleaner comparison
print(final_df.sort_values(["standards", "printer"]).to_string())

   printer  stl_texture  standards  mean_dist  std_error  sample_size
0    BAMBU            1      False      0.505      0.021            6
2    BAMBU            2      False      0.945      0.015            6
4    BAMBU            3      False      0.624      0.029            5
6    BAMBU            4      False      0.918      0.024            6
8    BAMBU            5      False      0.567      0.108            6
10   BAMBU            6      False      0.802      0.019            6
12   ENDER            1      False      1.278        NaN            1
13   ENDER            2      False      1.215      0.017            6
15   ENDER            3      False      1.073      0.043            5
17   ENDER            4      False      1.319      0.085            4
19   ENDER            5      False      1.121      0.076            4
21   RESIN            1      False      0.720      0.032            4
23   RESIN            2      False      0.767      0.036            5
25   RESIN          

In [6]:
# export
final_df.sort_values(["standards", "printer"]).to_csv("final_comp_result_under1.5.csv")